# ResNet ablations on a GPU (Kaggle or Colab)

This notebook runs the GPU part of the project: the 27 training runs (9 variants × 3 seeds).
Everything else (tests, gradient-flow analysis, plots) also runs on a laptop.

**Before you start**
- **Kaggle:** Settings → Accelerator → **GPU T4 ×2**, and Settings → **Internet on** (this needs a phone-verified account).
  For long runs, use **Save Version → Save & Run All (Commit)**. That runs in the background for up to 12 h, and everything
  written to `/kaggle/working` is kept as the notebook's output.
- **Colab:** Runtime → Change runtime type → **T4 GPU**. Runs are written to your Google Drive, so a disconnect loses at most
  one epoch: re-run the notebook and every run resumes from its checkpoint.

**Getting the code here:** set `REPO_URL` below to your GitHub repo, *or* upload `resnet-ablations.zip`
(Kaggle: as a Dataset; Colab: to Drive `MyDrive/`).

In [ ]:
import os, subprocess, sys, pathlib
REPO_URL = ""          # e.g. "https://github.com/<you>/resnet-ablations.git"  (leave empty to use the zip)
EPOCHS   = 60          # 182 = the paper's budget; 60 = the reduced budget (see README §3). Decide after the benchmark cell.

ON_KAGGLE = pathlib.Path("/kaggle").exists()
ON_COLAB  = "google.colab" in sys.modules
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
WORK = pathlib.Path("/kaggle/working" if ON_KAGGLE else "/content")
PROJ = WORK / "resnet-ablations"
if not PROJ.exists():
    if REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, str(PROJ)], check=True)
    elif ON_KAGGLE:
        # Kaggle extracts uploaded zips, and /kaggle/input is read-only: find the project folder and copy it.
        import shutil
        # If the zip was not extracted on upload, unzip it ourselves.
        hits = list(pathlib.Path("/kaggle/input").rglob("resnet/models.py"))
        if hits:
            shutil.copytree(hits[0].parent.parent, PROJ)
        else:
            z = next(pathlib.Path("/kaggle/input").rglob("resnet-ablations.zip"))
            subprocess.run(["unzip", "-q", str(z), "-d", str(WORK)], check=True)
    else:
        subprocess.run(["unzip", "-q", "/content/drive/MyDrive/resnet-ablations.zip", "-d", str(WORK)], check=True)
os.chdir(PROJ)
if ON_COLAB:   # keep runs on Drive so they survive disconnects
    persist = pathlib.Path("/content/drive/MyDrive/resnet-ablations-runs")
    persist.mkdir(parents=True, exist_ok=True)
    if not pathlib.Path("runs").exists():
        os.symlink(persist, "runs")
print(os.getcwd())

In [ ]:
!pip -q install pyyaml
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m scripts.download_data
!python -m pytest -q

## 1. Benchmark: measure, then pick the budget

This prints the measured seconds per epoch and the GPU-hours for each budget. Rough guide:
Kaggle gives ~30 GPU-hours/week and 12 h per session; free Colab is a few hours per day and disconnects.

In [ ]:
import torch
N_GPUS = torch.cuda.device_count()
!python -m scripts.benchmark --gpus {N_GPUS}

## 2. Run the ablations

With 2 GPUs, the run list is split into two shards, one process per GPU. Progress goes to `logs/shard*.log`.
Re-running this cell after an interruption resumes, and finished runs are skipped.

In [ ]:
pathlib.Path("logs").mkdir(exist_ok=True)
procs = []
for i in range(max(N_GPUS, 1)):
    log = open(f"logs/shard{i}.log", "a")
    cmd = [sys.executable, "-m", "scripts.run_ablation", "all", "--epochs", str(EPOCHS),
           "--shard", f"{i}/{max(N_GPUS, 1)}", "--device", f"cuda:{i}" if N_GPUS else "auto"]
    procs.append(subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT))
print("started", len(procs), "shard(s)")

In [ ]:
# Wait for all shards, printing the last line of each log every 5 minutes.
import time
while any(p.poll() is None for p in procs):
    time.sleep(300)
    for i in range(len(procs)):
        lines = pathlib.Path(f"logs/shard{i}.log").read_text().strip().splitlines()
        print(f"[shard {i}]", lines[-1] if lines else "(starting)")
print("exit codes:", [p.returncode for p in procs])
!python -m scripts.run_ablation all --epochs {EPOCHS} --list

## 3. Analyze and package the results

In [ ]:
!python -m scripts.analyze --runs runs/e{EPOCHS}
!zip -qr /tmp/results_e{EPOCHS}.zip runs/e{EPOCHS} results/e{EPOCHS} logs -x "*.pt"
!cp /tmp/results_e{EPOCHS}.zip {WORK}/
print(f"Download {WORK}/results_e{EPOCHS}.zip, unzip it in the project folder on your laptop,")
print(f"then run: python -m scripts.analyze --runs runs/e{EPOCHS}")

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(pathlib.Path(f"results/e{EPOCHS}/table.md").read_text()))
for f in ("degradation", "bn", "wd", "optim"):
    display(Image(f"results/e{EPOCHS}/{f}.png"))